In [ ]:
import pypsa
import plotly.express as px

network_path = "results/networks/base_s_40_elec_.nc"
n = pypsa.Network(network_path)

# 1. Regrouper par carrier via transposition (.T)
p_by_carrier = (
    n.generators_t.p.T
    .groupby(n.generators.carrier)
    .sum()
    .T / 1e3  # Conversion MW -> GW
)

# 2. Ajouter les unités de stockage si présentes
if not n.storage_units_t.p.empty:
    p_storage = (
        n.storage_units_t.p.T
        .groupby(n.storage_units.carrier)
        .sum()
        .T / 1e3
    )
    p_by_carrier = p_by_carrier.add(p_storage, fill_value=0)

# 3. Tracé interactif Plotly
fig = px.area(
    p_by_carrier,
    title="Hourly dispatch per technology (GW)",
    labels={"value": "Power (GW)", "snapshot": "Date", "variable": "Technology"}
    
)
fig.show()

In [ ]:
import glob
import re
import pandas as pd
import pypsa

# 1. Lister tous les fichiers réseaux résolus pour chaque wcf
files = sorted(
    glob.glob("results/networks/base_s_40__Co2L0-168h-lv1.25-wcf*_2050.nc")
)

caps_lw13 = {}
caps_lw25 = {}

for f in files:
    # Récupérer la valeur de wcf depuis le nom du fichier (ex: wcf0.7 -> 0.7)
    match = re.search(r"wcf([0-9\.]+)", f)
    if not match:
        continue
    wcf = float(match.group(1))

    n = pypsa.Network(f)

    # Récupérer p_nom_opt (en GW) par bus pour les deux technologies
    cap_col = "p_nom_opt" if "p_nom_opt" in n.generators.columns else "p_nom"

    type2 = (
        n.generators[n.generators.carrier == "onwind_type2"]
        .groupby("bus")[cap_col]
        .sum()
        / 1e3
    )
    type1 = (
        n.generators[n.generators.carrier == "onwind"]
        .groupby("bus")[cap_col]
        .sum()
        / 1e3
    )

    caps_lw13[wcf] = type2
    caps_lw25[wcf] = type1

# 2. Construire la matrice des capacités LW13 (lignes: bus, colonnes: wcf)
df_lw13 = pd.DataFrame(caps_lw13).sort_index(axis=1)
df_lw13.to_csv("results/lw13_capacity_by_cluster.csv")

# 3. Calculer le seuil de bascule (threshold) pour chaque cluster :
# C'est le wcf le plus élevé où la turbine LW13 est encore installée (> 50 MW = 0.05 GW)
thresholds = {}
for bus in df_lw13.index:
    series = df_lw13.loc[bus]
    built = series[series > 0.05].index
    thresholds[bus] = max(built) if len(built) > 0 else 0.0

df_summary = (
    pd.Series(thresholds, name="Break_Even_WCF")
    .sort_values(ascending=False)
    .to_frame()
)

print("=== SEUIL DE BASCULE DE LA LW13 PAR CLUSTER ===")
print(df_summary)

In [ ]:
import glob
import os
from pathlib import Path
import re
import geopandas as gpd
from matplotlib.collections import LineCollection
import matplotlib.colors as mcolors
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pypsa

# =====================================================================
# 0. ENSURE OUTPUT DIRECTORIES EXIST
# =====================================================================
output_dir = Path("results/plots")
output_dir.mkdir(parents=True, exist_ok=True)
Path("results").mkdir(parents=True, exist_ok=True)

# =====================================================================
# 1. LOAD NETWORKS & BUILD CAPACITY MATRICES
# =====================================================================
net_files = sorted(
    glob.glob("results/networks/base_s_40__Co2L0-168h-lv1.25-wcf*_2050.nc")
)
if not net_files:
    # Fallback search if path varies slightly
    net_files = sorted(glob.glob("results/**/base_s_40*wcf*.nc", recursive=True))

if not net_files:
    raise FileNotFoundError(
        "No solved network files matching 'wcf' found in results/networks/!"
    )

print(f"Found {len(net_files)} solved network scenarios to process.")

lw13_by_cluster = {}
total_lw13_gw = {}
total_lw25_gw = {}
ref_network = None

for path in net_files:
    match = re.search(r"wcf([0-9\.]+)", path)
    if not match:
        continue
    wcf = float(match.group(1))

    n = pypsa.Network(path)
    if ref_network is None:
        ref_network = n  # Reference network for coordinates and topology

    # Determine capacity column (p_nom_opt if solved, else p_nom)
    cap_col = "p_nom_opt" if "p_nom_opt" in n.generators.columns else "p_nom"

    # LW13 (onwind_type2) capacity per bus (GW)
    mask_type2 = n.generators.carrier == "onwind_type2"
    if mask_type2.any():
        lw13_series = (
            n.generators[mask_type2].groupby("bus")[cap_col].sum() / 1e3
        )
    else:
        lw13_series = pd.Series(0.0, index=n.buses.index)

    lw13_by_cluster[wcf] = lw13_series
    total_lw13_gw[wcf] = lw13_series.sum()

    # Standard LW25 (onwind) capacity per bus (GW)
    mask_type1 = n.generators.carrier == "onwind"
    if mask_type1.any():
        lw25_series = (
            n.generators[mask_type1].groupby("bus")[cap_col].sum() / 1e3
        )
    else:
        lw25_series = pd.Series(0.0, index=n.buses.index)

    total_lw25_gw[wcf] = lw25_series.sum()

df_lw13 = pd.DataFrame(lw13_by_cluster).sort_index(axis=1)
df_total = pd.DataFrame(
    {"LW13 (onwind_type2)": total_lw13_gw, "LW25 (onwind)": total_lw25_gw}
).sort_index()

# =====================================================================
# 2. COMPUTE BREAK-EVEN THRESHOLDS PER CLUSTER
# =====================================================================
# Break-even threshold = maximum wcf where LW13 is installed (> 50 MW = 0.05 GW)
thresholds = {}
for bus in df_lw13.index:
    series = df_lw13.loc[bus]
    built = series[series > 0.05].index
    thresholds[bus] = max(built) if len(built) > 0 else 0.0

df_thresholds = (
    pd.Series(thresholds, name="Break_Even_WCF")
    .sort_values(ascending=False)
    .to_frame()
)
df_thresholds.to_csv("results/cluster_break_even_wcf.csv")

print("\n=== BREAK-EVEN COST FACTORS BY CLUSTER (Top 10) ===")
print(df_thresholds.head(10))

# =====================================================================
# 3. PLOT 1: LW13 vs LW25 CAPACITY AS A FUNCTION OF WCF
# =====================================================================
plt.figure(figsize=(9, 5.5))
plt.plot(
    df_total.index,
    df_total["LW13 (onwind_type2)"],
    marker="o",
    linewidth=2.5,
    color="#0d9488",
    label="LW13 (Low-Wind 13 m/s)",
)
plt.plot(
    df_total.index,
    df_total["LW25 (onwind)"],
    marker="s",
    linewidth=2,
    linestyle="--",
    color="#f59e0b",
    label="LW25 (Standard 25 m/s)",
)

plt.title("European Installed Wind Capacity vs. LW13 Cost Factor (wcf)", pad=12)
plt.xlabel("Wind Cost Factor (wcf)")
plt.ylabel("Installed Capacity (GW)")
plt.grid(True, linestyle=":", alpha=0.6)
plt.legend(frameon=True)
plt.tight_layout()

# Save figure 1 safely
plt.savefig(
    output_dir / "lw13_capacity_vs_wcf.png", dpi=300, bbox_inches="tight"
)
plt.show()

# =====================================================================
# 4. PLOT 2: GEOGRAPHIC MAP OF THRESHOLDS PER CLUSTER
# =====================================================================
fig, ax = plt.subplots(figsize=(11, 10))

# Europe background borders
url = "https://raw.githubusercontent.com/nvkelso/natural-earth-vector/master/geojson/ne_110m_admin_0_countries.geojson"
try:
    world = gpd.read_file(url)
    world.cx[-15:35, 34:72].plot(
        ax=ax, color="#f8fafc", edgecolor="#cbd5e1", linewidth=0.8
    )
except Exception:
    pass

# Grid lines (AC and DC links)
bus_x = ref_network.buses.x.to_dict()
bus_y = ref_network.buses.y.to_dict()

segments = [
    [(bus_x[b0], bus_y[b0]), (bus_x[b1], bus_y[b1])]
    for b0, b1 in zip(ref_network.lines.bus0, ref_network.lines.bus1)
    if b0 in bus_x and b1 in bus_x
]
ax.add_collection(
    LineCollection(
        segments,
        colors="#94a3b8",
        linewidths=0.9,
        alpha=0.5,
        label="Transmission Grid",
    )
)

# Align cluster threshold values with bus positions
plot_df = ref_network.buses[["x", "y"]].copy()
plot_df["threshold"] = df_thresholds["Break_Even_WCF"].reindex(
    plot_df.index, fill_value=0.0
)

# Colormap settings
cmap = plt.cm.viridis
vmax_val = max(df_thresholds["Break_Even_WCF"].max(), 0.1)
norm = mcolors.Normalize(vmin=0.0, vmax=vmax_val)

scatter = ax.scatter(
    plot_df.x,
    plot_df.y,
    c=plot_df.threshold,
    s=plot_df.threshold * 250 + 60,
    cmap=cmap,
    norm=norm,
    edgecolors="black",
    linewidth=0.8,
    zorder=4,
)

# Add cluster labels
for idx, row in plot_df.iterrows():
    if row.threshold > 0:
        ax.annotate(
            f"{idx}\n({row.threshold:.2f})",
            (row.x, row.y),
            xytext=(3, 3),
            textcoords="offset points",
            fontsize=7,
            weight="bold",
        )

cbar = plt.colorbar(scatter, ax=ax, fraction=0.03, pad=0.03)
cbar.set_label("Break-even Cost Factor (wcf threshold)", rotation=270, labelpad=15)

# View limits
margin = 2.0
ax.set_xlim(ref_network.buses.x.min() - margin, ref_network.buses.x.max() + margin)
ax.set_ylim(ref_network.buses.y.min() - margin, ref_network.buses.y.max() + margin)
ax.set_title(
    "LW13 Break-Even Cost Factor Threshold Across 40 European Clusters", pad=12
)
ax.set_xlabel("Longitude (°E)")
ax.set_ylabel("Latitude (°N)")
ax.grid(True, linestyle=":", alpha=0.4)

plt.tight_layout()

# Save figure 2 safely
plt.savefig(
    output_dir / "map_thresholds_clusters.png", dpi=300, bbox_inches="tight"
)
plt.show()

print("\nAll plots and summary tables saved successfully to:")
print(" - results/cluster_break_even_wcf.csv")
print(" - results/plots/lw13_capacity_vs_wcf.png")
print(" - results/plots/map_thresholds_clusters.png")

In [4]:
import os
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pypsa

# Ensure output directory exists
Path("results/plots").mkdir(parents=True, exist_ok=True)


def setup_operational_network(net_path, voll=3000.0):
    """Loads a solved network, locks all capacities, disables cyclic constraints,

    and adds load-shedding generators for resilience tracking.
    """
    print(f"Loading network: {net_path}")
    n = pypsa.Network(net_path)

    # 1. Lock all generator and link capacities to optimal sizes
    for c in ["generators", "links", "storage_units", "stores"]:
        df = getattr(n, c)
        if "p_nom_opt" in df.columns:
            df["p_nom"] = df["p_nom_opt"]
        if "p_nom_extendable" in df.columns:
            df["p_nom_extendable"] = False
        if "e_nom_opt" in df.columns:
            df["e_nom"] = df["e_nom_opt"]
        if "e_nom_extendable" in df.columns:
            df["e_nom_extendable"] = False

    # 2. Disable cyclic storage constraints for rolling horizon
    # (prevents artificial end-of-window discharge requirements)
    if not n.storage_units.empty:
        n.storage_units["cyclic_state_of_charge"] = False
        n.storage_units["state_of_charge_initial"] = (
            n.storage_units["p_nom"] * 0.5
        )

    if not n.stores.empty:
        n.stores["e_cyclic"] = False
        n.stores["e_initial"] = n.stores["e_nom"] * 0.5

    # 3. Add load shedding generators to each bus
    shedding_names = [f"load_shedding_{b}" for b in n.buses.index]
    # Remove existing ones if re-running
    existing = [g for g in shedding_names if g in n.generators.index]
    if existing:
        n.mremove("Generator", existing)

    n.add(
        "Generator",
        shedding_names,
        bus=n.buses.index,
        carrier="load_shedding",
        p_nom=1e6,  # Unlimited capacity
        marginal_cost=voll,  # Value of Lost Load (€/MWh)
    )

    return n


def run_rolling_horizon(n, window_hours=336):
    """Executes a rolling horizon operational dispatch with a fixed window size

    (default: 336 hours = 2 weeks), transferring state of charge between windows.
    """
    snapshots = n.snapshots
    total_hours = len(snapshots)
    windows = [
        snapshots[i : i + window_hours]
        for i in range(0, total_hours, window_hours)
    ]
    print(
        f"Starting rolling horizon: {len(windows)} windows of {window_hours} hours each."
    )

    for i, w in enumerate(windows):
        print(f" -> Solving window {i + 1}/{len(windows)} ({w[0]} to {w[-1]})...")

        # Solve dispatch for current window using fast barrier settings
        status, condition = n.optimize(
            snapshots=w,
            solver_name="gurobi",
            solver_options={
                "Method": 2,  # Barrier
                "Crossover": 0,  # Fast convergence without crossover
                "BarConvTol": 1e-4,
                "OutputFlag": 0,  # Silent solver log
            },
        )

        if condition != "optimal":
            print(f"Warning: Window {i + 1} finished with status: {condition}")

        # Carry forward storage states to next window
        if i < len(windows) - 1:
            last_snp = w[-1]

            # Storage units (e.g. hydro reservoirs)
            if not n.storage_units.empty:
                soc_last = n.storage_units_t.state_of_charge.loc[last_snp]
                n.storage_units["state_of_charge_initial"] = soc_last

            # Stores (e.g. battery, H2, heat tanks)
            if not n.stores.empty:
                e_last = n.stores_t.e.loc[last_snp]
                n.stores["e_initial"] = e_last

    return n


def evaluate_resilience(n, label="Case"):
    """Extracts load shedding and calculates LOLE, EENS, and wind generation."""
    shed_gens = [g for g in n.generators.index if "load_shedding" in g]
    shed_p = n.generators_t.p[shed_gens].sum(axis=1)  # MW per hour

    # Loss of Load Expectation (hours with deficit > 1 MW)
    lole_hours = int((shed_p > 1.0).sum())

    # Expected Energy Not Served (MWh converted to GWh)
    eens_gwh = float(shed_p.sum() / 1e3)

    # Generation summary
    p_by_carrier = (
        n.generators_t.p.T.groupby(n.generators.carrier).sum().sum(axis=1) / 1e3
    )  # GWh
    lw13_gen = float(p_by_carrier.get("onwind_type2", 0.0))
    lw25_gen = float(p_by_carrier.get("onwind", 0.0))

    metrics = {
        "Scenario": label,
        "LOLE [hours]": lole_hours,
        "EENS [GWh]": round(eens_gwh, 2),
        "LW13 Generation [GWh]": round(lw13_gen, 2),
        "LW25 Generation [GWh]": round(lw25_gen, 2),
    }

    return metrics, shed_p

In [ ]:
# Select the two reference solved networks from Step 1
# Case 1: Baseline system (dominated by standard LW25)
path_baseline = (
    "results/networks/base_s_40__Co2L0-168h-lv1.25-wcf1.0_2050.nc"  # or wcf1.1
)

# Case 2: Low-Wind hybrid system (high penetration of LW13)
path_hybrid = (
    "results/networks/base_s_40__Co2L0-168h-lv1.25-wcf0.7_2050.nc"  # or wcf0.8
)

# 1. Run Baseline
n_base = setup_operational_network(path_baseline)
n_base = run_rolling_horizon(n_base, window_hours=336)
metrics_base, shed_base = evaluate_resilience(
    n_base, label="Baseline (Standard LW25)"
)

# 2. Run Hybrid LW13
n_hybrid = setup_operational_network(path_hybrid)
n_hybrid = run_rolling_horizon(n_hybrid, window_hours=336)
metrics_hybrid, shed_hybrid = evaluate_resilience(
    n_hybrid, label="Hybrid (LW13 + LW25)"
)

# 3. Present Results Table
df_resilience = pd.DataFrame([metrics_base, metrics_hybrid]).set_index(
    "Scenario"
)
df_resilience.to_csv("results/resilience_metrics_comparison.csv")
print("\n" + "=" * 55)
print("       SYSTEM RESILIENCE COMPARISON (2-WEEK ROLLING)")
print("=" * 55)
print(df_resilience)
print("=" * 55)

# 4. Plot Load Shedding Timeline
plt.figure(figsize=(11, 4.5))
plt.plot(
    shed_base.index,
    shed_base,
    label=f"Baseline (EENS = {metrics_base['EENS [GWh]']} GWh)",
    color="#ef4444",
    alpha=0.8,
)
plt.plot(
    shed_hybrid.index,
    shed_hybrid,
    label=f"Hybrid LW13 (EENS = {metrics_hybrid['EENS [GWh]']} GWh)",
    color="#0d9488",
    alpha=0.8,
)

plt.title("Hourly Load Shedding (Unserved Demand) under 2-Week Rolling Horizon")
plt.xlabel("Snapshot")
plt.ylabel("Unserved Power [MW]")
plt.grid(True, linestyle=":", alpha=0.5)
plt.legend(frameon=True)
plt.tight_layout()
plt.savefig("results/plots/resilience_load_shedding_comparison.png", dpi=300)
plt.show()

In [ ]:
print(n.buses_t.marginal_price)

In [5]:
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pypsa

Path("results/plots").mkdir(parents=True, exist_ok=True)


def apply_frozen_capacities_clean(n_operational, n_design):
    """Fige les capacités optimales dimensionnées en 2050 dans le réseau opérationnel 1968

    et ajoute des générateurs de délestage d'urgence à 3 000 €/MWh.
    """
    # 1. GÉNÉRATEURS (éolien, solaire, etc.)
    n_operational.generators["p_nom"] = 0.0
    n_operational.generators["p_nom_extendable"] = False
    common_gens = n_operational.generators.index.intersection(
        n_design.generators.index
    )
    n_operational.generators.loc[common_gens, "p_nom"] = (
        n_design.generators.loc[common_gens, "p_nom_opt"]
    )

    # 2. LIGNES ÉLECTRIQUES
    common_lines = n_operational.lines.index.intersection(n_design.lines.index)
    if "s_nom_opt" in n_design.lines.columns:
        n_operational.lines.loc[common_lines, "s_nom"] = n_design.lines.loc[
            common_lines, "s_nom_opt"
        ]
    n_operational.lines["s_nom_extendable"] = False

    # 3. LIENS DE CONVERSION SECTORIELLE (HVDC, pompes à chaleur, H2, etc.)
    if not n_operational.links.empty and not n_design.links.empty:
        n_operational.links["p_nom"] = 0.0
        n_operational.links["p_nom_extendable"] = False
        common_links = n_operational.links.index.intersection(
            n_design.links.index
        )
        if "p_nom_opt" in n_design.links.columns:
            n_operational.links.loc[common_links, "p_nom"] = (
                n_design.links.loc[common_links, "p_nom_opt"]
            )

    # 4. STORAGE UNITS (Hydroélectricité / STEP)
    if not n_operational.storage_units.empty:
        n_operational.storage_units["p_nom"] = 0.0
        n_operational.storage_units["p_nom_extendable"] = False
        common_su = n_operational.storage_units.index.intersection(
            n_design.storage_units.index
        )
        if "p_nom_opt" in n_design.storage_units.columns:
            n_operational.storage_units.loc[common_su, "p_nom"] = (
                n_design.storage_units.loc[common_su, "p_nom_opt"]
            )
        n_operational.storage_units["cyclic_state_of_charge"] = False
        n_operational.storage_units["state_of_charge_initial"] = (
            n_operational.storage_units["p_nom"] * 0.15
        )

    # 5. STORES (Batteries, stockages H2, réservoirs de chaleur)
    if not n_operational.stores.empty and not n_design.stores.empty:
        n_operational.stores["e_nom"] = 0.0
        n_operational.stores["e_nom_extendable"] = False
        common_stores = n_operational.stores.index.intersection(
            n_design.stores.index
        )
        if "e_nom_opt" in n_design.stores.columns:
            n_operational.stores.loc[common_stores, "e_nom"] = (
                n_design.stores.loc[common_stores, "e_nom_opt"]
            )
        n_operational.stores["e_cyclic"] = False
        n_operational.stores["e_initial"] = (
            n_operational.stores["e_nom"] * 0.15
        )

    # 6. GÉNÉRATEURS DE DÉLESTAGE (VOLL = 3000 €/MWh sur chaque bus)
    shed_names = [f"load_shed_{b}" for b in n_operational.buses.index]
    existing = [g for g in shed_names if g in n_operational.generators.index]
    if existing:
        n_operational.mremove("Generator", existing)

    n_operational.add(
        "Generator",
        shed_names,
        bus=n_operational.buses.index,
        carrier="load_shedding",
        p_nom=1e6,
        marginal_cost=3000.0,
    )
    return n_operational


def run_rolling_horizon(n, window_hours=336):
    """Résout le dispatch opérationnel par bloc temporel (336 h = 2 semaines)

    en propageant l'état des stockages d'une fenêtre à la suivante.
    """
    snapshots = n.snapshots
    windows = [
        snapshots[i : i + window_hours]
        for i in range(0, len(snapshots), window_hours)
    ]
    print(f"Simulation glissante lancée : {len(windows)} fenêtres de 2 semaines.")

    for i, w in enumerate(windows):
        status, condition = n.optimize(
            snapshots=w,
            solver_name="gurobi",
            solver_options={
                "Method": 2,  # Barrière
                "Crossover": 0,
                "BarConvTol": 1e-4,
                "OutputFlag": 0,
            },
        )
        # Propagation de l'état des stocks vers la fenêtre suivante
        if i < len(windows) - 1:
            last_snp = w[-1]
            if not n.storage_units.empty:
                n.storage_units["state_of_charge_initial"] = (
                    n.storage_units_t.state_of_charge.loc[last_snp]
                )
            if not n.stores.empty:
                n.stores["e_initial"] = n.stores_t.e.loc[last_snp]

    return n


def evaluate_resilience(n, label="Scénario"):
    """Extrait le délestage (kW/MWh), les heures de perte de charge et la production éolienne."""
    shed_gens = [g for g in n.generators.index if "load_shed" in g]
    shed_p = n.generators_t.p[shed_gens].sum(axis=1)  # MW

    # Seuil à 0.01 MW (10 kW) pour éliminer le bruit numérique
    lole_hours = int((shed_p > 0.01).sum())
    eens_mwh = float(shed_p.sum())

    # Production réelle des éoliennes en GWh
    p_by_carrier = (
        n.generators_t.p.T.groupby(n.generators.carrier).sum().sum(axis=1) / 1e3
    )
    lw13_gen = float(p_by_carrier.get("onwind_type2", 0.0))
    lw25_gen = float(p_by_carrier.get("onwind", 0.0))

    metrics = {
        "Scénario": label,
        "LOLE [heures]": lole_hours,
        "EENS [MWh]": round(eens_mwh, 2),
        "Pic Délester [MW]": round(float(shed_p.max()), 2),
        "LW13 Production [GWh]": round(lw13_gen, 1),
        "LW25 Production [GWh]": round(lw25_gen, 1),
    }
    return metrics, shed_p

In [ ]:
# =====================================================================
# 1. CHARGEMENT DES DESIGNS 2050 (DIMENSIONNÉS SUR 2013)
# =====================================================================
print("Chargement des réseaux sectoriels résolus en 2013...")
n_design_base = pypsa.Network(
    "results/networks/base_s_40__Co2L0-168h-lv1.25-wcf1.0_2050.nc"
)
n_design_hybrid = pypsa.Network(
    "results/networks/base_s_40__Co2L0-168h-lv1.25-wcf0.7_2050.nc"
)

# 2. NEUTRALISATION STRICTE DE LA LW13 DANS LA BASELINE
mask_type2 = n_design_base.generators.carrier == "onwind_type2"
mask_type1 = n_design_base.generators.carrier == "onwind"
if mask_type2.any():
    cap_type2_mw = n_design_base.generators.loc[mask_type2, "p_nom_opt"].values
    n_design_base.generators.loc[mask_type1, "p_nom_opt"] += cap_type2_mw
    n_design_base.generators.loc[mask_type2, "p_nom_opt"] = 0.0

# =====================================================================
# 3. CHARGEMENT DU RÉSEAU OPÉRATIONNEL CIBLE POUR 1968
# =====================================================================
# Utiliser le réseau sectoriel préparé de 1968 s'il existe, sinon le réseau elec
path_1968 = "resources/networks/base_s_40_elec_1968.nc"

print(f"Chargement du réseau météo cible : {path_1968}")
n_stress_base = pypsa.Network(path_1968)
n_stress_hybrid = pypsa.Network(path_1968)

# 4. APPLICATION DU VERROUILLAGE DES CAPACITÉS
n_stress_base = apply_frozen_capacities_clean(n_stress_base, n_design_base)
n_stress_hybrid = apply_frozen_capacities_clean(n_stress_hybrid, n_design_hybrid)

# CONTRÔLE DES PARCS ÉOLIENS AVANT CALCUL
print("\n--- CAPACITÉS FIGÉES POUR LE TEST 1968 ---")
base_lw13 = n_stress_base.generators[
    n_stress_base.generators.carrier == "onwind_type2"
]["p_nom"].sum()
base_lw25 = n_stress_base.generators[
    n_stress_base.generators.carrier == "onwind"
]["p_nom"].sum()
hyb_lw13 = n_stress_hybrid.generators[
    n_stress_hybrid.generators.carrier == "onwind_type2"
]["p_nom"].sum()
hyb_lw25 = n_stress_hybrid.generators[
    n_stress_hybrid.generators.carrier == "onwind"
]["p_nom"].sum()

print(f"Baseline -> LW13: {base_lw13:.1f} MW | LW25: {base_lw25:.1f} MW")
print(f"Hybride  -> LW13: {hyb_lw13:.1f} MW | LW25: {hyb_lw25:.1f} MW")

In [ ]:
# =====================================================================
# RÉSOLUTION GLISSANTE (336h) SUR L'ANNÉE MÉTÉO 1968
# =====================================================================
print("\n--- Lancement du Rolling Horizon : Baseline pure LW25 ---")
n_stress_base = run_rolling_horizon(n_stress_base, window_hours=336)

print("\n--- Lancement du Rolling Horizon : Hybride LW13 + LW25 ---")
n_stress_hybrid = run_rolling_horizon(n_stress_hybrid, window_hours=336)

# Évaluation des résultats
metrics_base, shed_base = evaluate_resilience(
    n_stress_base, label="1968 Baseline (Pure LW25)"
)
metrics_hyb, shed_hyb = evaluate_resilience(
    n_stress_hybrid, label="1968 Hybride (LW13 + LW25)"
)

# Affichage du tableau comparatif
df_resilience = pd.DataFrame([metrics_base, metrics_hyb]).set_index("Scénario")
df_resilience.to_csv("results/resilience_metrics_1968.csv")

print("\n" + "=" * 65)
print("     RÉSULTATS DE RÉSILIENCE SUR L'ANNÉE HISTORIQUE 1968")
print("=" * 65)
print(df_resilience)
print("=" * 65)


In [ ]:
import pandas as pd
import pypsa

# Charger l'un de vos réseaux résolus de 2050
n = pypsa.Network(
    "results/networks/base_s_40__Co2L0-168h-lv1.25-wcf1.0_2050.nc"
)

# 1. Vérification au niveau GLOBAL (le Line Volume total)
# Le Line Volume est pondéré par la longueur des lignes (en MVA * km)
initial_line_volume = (n.lines.s_nom * n.lines.length).sum()
optimal_line_volume = (n.lines.s_nom_opt * n.lines.length).sum()

global_ratio = optimal_line_volume / initial_line_volume

print("=== VÉRIFICATION GLOBALE DU LINE VOLUME (lv) ===")
print(f"Volume initial : {initial_line_volume / 1e3:.1f} GVA·km")
print(f"Volume optimal : {optimal_line_volume / 1e3:.1f} GVA·km")
print(f"Ratio global   : {global_ratio:.4f} (Doit être <= 1.2500)")

# 2. Vérification LIGNE PAR LIGNE
lines_df = pd.DataFrame(
    {
        "s_nom [MVA]": n.lines.s_nom,
        "s_nom_opt [MVA]": n.lines.s_nom_opt,
        "Ratio (opt/init)": (n.lines.s_nom_opt / n.lines.s_nom).round(3),
        "Extension [MVA]": (n.lines.s_nom_opt - n.lines.s_nom).round(1),
        "Longueur [km]": n.lines.length.round(1),
    }
)

print("\n=== LIGNES AVEC LE PLUS FORT TAUX D'EXPANSION ===")
print(lines_df.sort_values(by="Ratio (opt/init)", ascending=False).head(10))